<a href="https://colab.research.google.com/github/Niladri-Baksi/SpeakEdge/blob/main/SpeakEdge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modules
1. Setup & Install
2. Audio Input (upload)
3. Speech-to-Text (Whisper)
4. Feature Extraction
5. Scoring Algorithm
6. Feedback + Recommendations

### Module 1

In [1]:
!pip install openai-whisper
!pip install transformers
!pip install sentence-transformers
!pip install librosa
!pip install soundfile

!pip install --upgrade transformers
from transformers import pipeline

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 22.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=bccfcd9714864def1aad587cdf2dc1cb11be2f1336d5a30a9986493adfd321f4
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 74.6 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
import whisper
import librosa
import numpy as np
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline

### Module 2

In [3]:
from google.colab import files
uploaded = files.upload()
audio_file = list(uploaded.keys())[0]
print(audio_file)

Saving zuck-nil.wav to zuck-nil.wav
zuck-nil.wav


### Module 3

In [4]:
model = whisper.load_model("base")

100%|███████████████████████████████████████| 139M/139M [00:02<00:00, 55.1MiB/s]


In [5]:
#STT
result = model.transcribe(audio_file)
transcript = result["text"]

print("Transcript:")
print(transcript)

Transcript:
 Hey everyone, it is Lama 4 day. Our goal is to build the world's leading AI, open-source it, and make it universally accessible so that everyone in the world benefits. And I've said for a while that I think that open-source AI is going to become the leading models. And with Lama 4, this is starting to happen. Mene AI is getting a big upgrade today. So if you want to try Lama 4, you can use Mene AI in WhatsApp, Messenger, or Instagram direct, or you can go to our website at meta.ai. Today we are dropping the first two open-source Lama 4 models, where we've got two more on the way. The first model is Lama 4 Scout. It is extremely fast, natively multimodal. It has an industry leading nearly infinite 10 million token context length, and it is designed to run on a single GPU. It is 17 billion parameters by 16 experts, and it is by far the highest performing small model in its class. The second model is Lama 4 Maverick, the workhorse. It beats GPT-40 in Gemini Flash 2 on all ben

### Module 4

In [6]:
#Basic text features

import re

words = transcript.split()
total_words = len(words)
unique_words = len(set(words))

print("Total words:", total_words)
print("Unique words:", unique_words)

Total words: 378
Unique words: 202


In [7]:
#Filler word detection
filler_words = ["um", "uh", "like", "you know", "basically"]

filler_count = sum(transcript.lower().count(f) for f in filler_words)

print("Filler count:", filler_count)

Filler count: 0


In [8]:
#Speech duration + WPM
audio, sr = librosa.load(audio_file)
duration = librosa.get_duration(y=audio, sr=sr)

wpm = (total_words / duration) * 60

print("Duration:", duration)
print("Words per minute:", wpm)

Duration: 138.18
Words per minute: 164.1337386018237


In [9]:
# #Grammar checking (basic)
# grammar_pipeline = pipeline("text2text-generation", model="vennify/t5-base-grammar-correction")

# corrected = grammar_pipeline(transcript, max_length=512)[0]['generated_text']

# print("Corrected Text:")
# print(corrected)

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("vennify/t5-base-grammar-correction")
model = AutoModelForSeq2SeqLM.from_pretrained("vennify/t5-base-grammar-correction")

input_text = "fix grammar: " + transcript

inputs = tokenizer(input_text, return_tensors="pt")
outputs = model.generate(**inputs, max_length=512)

corrected = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(corrected)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

We are dropping the first two open-source Lama 4 models, and we have two more on the way. The first one is Lama 4 Reasoning, and we will share more about it soon.


In [10]:
#Coherence
model_embed = SentenceTransformer('all-MiniLM-L6-v2')

question = "Tell me about yourself"

emb1 = model_embed.encode(question, convert_to_tensor=True)
emb2 = model_embed.encode(transcript, convert_to_tensor=True)

similarity = util.cos_sim(emb1, emb2).item()

print("Similarity score:", similarity)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Similarity score: 0.04507822543382645
